# Recurrent Networks for Disease Progression

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 4: Recurrent Neural Networks — Sequence Modelling, Time-Series**

ChestX-ray14 carries `Patient ID` and `Follow-up #`, and averages 3–4 studies per
patient. Those columns turn a static image dataset into **longitudinal sequences**,
which is what makes a recurrent model clinically meaningful here rather than a
syllabus box to tick.

The task: given a patient's prior studies, predict the pathology state at the next
visit. A CNN encodes each visit into an embedding; a GRU reads the sequence.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. Building patient timelines

In [ ]:
def build_sequences(df, min_visits=2, max_len=6):
    """Group studies into per-patient sequences ordered by follow-up number."""
    df = df.sort_values(["Patient ID", "Follow-up #"])
    sequences = []
    for pid, g in df.groupby("Patient ID"):
        if len(g) < min_visits: continue
        g = g.head(max_len)
        sequences.append({
            "patient_id": pid,
            "images": g["Image Index"].tolist(),
            "labels": g[PATHOLOGIES].values.astype("float32"),
            "ages": g["Patient Age"].values.astype("float32"),
            "n_visits": len(g),
        })
    return sequences

def describe(seqs):
    lengths = [s["n_visits"] for s in seqs]
    print(f"patients with >=2 visits : {len(seqs):,}")
    print(f"mean visits              : {np.mean(lengths):.2f}")
    print(f"max visits (capped)      : {max(lengths)}")
    # How often does the label set actually change between visits? If it never
    # changed, the sequence task would be trivial and not worth modelling.
    changed = sum(
        1 for s in seqs if not np.array_equal(s["labels"][0], s["labels"][-1])
    )
    print(f"patients whose findings change: {changed:,} ({changed/len(seqs):.1%})")
    return lengths

print("Sequence builders ready.")

## 2. Padded batching

Sequences have different lengths, so they must be padded — and the padding must be masked, or the model learns from timesteps that do not exist.

In [ ]:
def collate_sequences(batch, max_len=6, dim=1024):
    """Pad to a common length and return the true lengths for masking."""
    B = len(batch)
    x = torch.zeros(B, max_len, dim)
    y = torch.zeros(B, 14)
    lengths = torch.zeros(B, dtype=torch.long)
    for i, s in enumerate(batch):
        n = min(len(s["embeddings"]) - 1, max_len)   # last visit is the target
        x[i, :n] = torch.as_tensor(s["embeddings"][:n])
        y[i] = torch.as_tensor(s["labels"][n])
        lengths[i] = n
    return x, y, lengths

class ProgressionGRU(nn.Module):
    def __init__(self, input_dim=1024, hidden=256, layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, layers, batch_first=True,
                          dropout=dropout if layers > 1 else 0.0)
        self.attn = nn.Linear(hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, 14))

    def forward(self, x, lengths):
        out, _ = self.gru(x)
        scores = self.attn(out).squeeze(-1)
        # Mask BEFORE softmax. Without this, padded timesteps receive
        # attention mass and the model learns from data that is not there.
        mask = torch.arange(out.size(1), device=x.device)[None] >= lengths[:, None]
        weights = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=1)
        context = torch.bmm(weights.unsqueeze(1), out).squeeze(1)
        return self.head(context), weights

m = ProgressionGRU()
xb, lb = torch.randn(4, 6, 1024), torch.tensor([6, 4, 2, 1])
logits, w = m(xb, lb)
print("logits", tuple(logits.shape), "| attention rows sum to 1:",
      torch.allclose(w.sum(1), torch.ones(4), atol=1e-5))
print("padded steps get zero attention:", torch.allclose(w[3, 1:], torch.zeros(5), atol=1e-6))

## 3. Baseline comparison

A sequence model must beat 'assume nothing changes'. Clinical states are persistent, so that baseline is strong and easy to lose to.

In [ ]:
def persistence_baseline(sequences):
    """Predict the next visit's labels as identical to the previous visit."""
    from sklearn.metrics import roc_auc_score
    preds, truth = [], []
    for s in sequences:
        if len(s["labels"]) < 2: continue
        preds.append(s["labels"][-2]); truth.append(s["labels"][-1])
    preds, truth = np.array(preds), np.array(truth)
    aucs = [roc_auc_score(truth[:, k], preds[:, k])
            for k in range(14) if 0 < truth[:, k].sum() < len(truth)]
    print(f"Persistence baseline macro AUROC: {np.mean(aucs):.4f}")
    print("Any recurrent model that does not beat this has learned nothing")
    print("about progression — it has only learned that findings persist.")
    return np.mean(aucs)

print("Baseline ready.")

---

### References for this notebook

- Lipton, Z. C. et al. (2015). A critical review of RNNs for sequence learning. arXiv:1506.00019.
- Graves, A., Mohamed, A. & Hinton, G. (2013). Speech recognition with deep RNNs. *ICASSP*.
- Bai, S., Kolter, J. Z. & Koltun, V. (2018). An empirical evaluation of generic convolutional and recurrent networks. arXiv:1803.01271.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
